# Phase 3.6 — BBA Training on Colab GPU

Trains `TangentScoreModel(256×4)` on 63k BBA frames with online Riemannian DSM.

**Before running**: select a GPU runtime (Runtime → Change runtime type → T4 / A100).  
**Repo path**: set `REPO_ROOT` in cell 0 to wherever the repo lives on the Colab machine.

## 0. Paths and dependencies

In [26]:
import subprocess, sys, os

# ── Mount Google Drive (data lives there, uploaded via rclone) ────────────────
from google.colab import drive
drive.mount("/content/drive")
# ─────────────────────────────────────────────────────────────────────────────

# ── Set this to wherever the repo lives on the Colab machine ─────────────────
REPO_ROOT = "/content"   # <-- edit if needed
# ─────────────────────────────────────────────────────────────────────────────

SRC_DIR         = os.path.join(REPO_ROOT, "riemannian-scoremd", "src")
PROCESSED_DIR   = "/content/drive/MyDrive/thesis-data/processed"        # bba_ref.npy, bba_test.npy
PRECOMPUTED_DIR = "/content/drive/MyDrive/thesis-data/precomputed/bba"  # noised_r00..r09.npz
CKPT_DIR        = "/content/runs/bba_phase36"

sys.path.insert(0, SRC_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "flax==0.10.4", "optax"], check=True)

print(f"src exists        : {os.path.isdir(SRC_DIR)}")
print(f"processed exists  : {os.path.isdir(PROCESSED_DIR)}")
precomp_files = sorted(os.listdir(PRECOMPUTED_DIR)) if os.path.isdir(PRECOMPUTED_DIR) else "NOT FOUND"
print(f"precomputed files : {precomp_files}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
src exists        : True
processed exists  : True
precomputed files : ['noised_r00.npz', 'noised_r01.npz', 'noised_r02.npz', 'noised_r03.npz', 'noised_r04.npz', 'noised_r05.npz', 'noised_r06.npz', 'noised_r07.npz', 'noised_r08.npz', 'noised_r09.npz']


In [27]:
# Check GPU and JAX
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
print("GPU     :", result.stdout.strip() or "NONE — change runtime type!")

import jax
print("JAX     :", jax.__version__)
print("Devices :", jax.devices())

GPU     : NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
JAX     : 0.7.2
Devices : [CudaDevice(id=0)]


In [28]:
from manifold.pointcloud_jax import ShapeManifold
from diffusion.manifold_sde import ManifoldVP
from models.tangent_mlp import TangentScoreModel
from training.score_loss import prepare_batch, prepare_batch_vmapped
from training.train_manifold import train, train_from_precomputed
print("Imports OK")

ImportError: cannot import name 'prepare_batch_vmapped' from 'training.score_loss' (/content/riemannian-scoremd/src/training/score_loss.py)

## 1. Load data

In [ ]:
import numpy as np

test_data = np.load(os.path.join(PROCESSED_DIR, "bba_test.npy"))   # (7000, 28, 3)
x_ref     = np.load(os.path.join(PROCESSED_DIR, "bba_ref.npy"))    # (28, 3)

n, d = x_ref.shape
print(f"Test : {test_data.shape}")
print(f"n={n} Cα, nd={n*d}")

# Peek at one precomputed file to confirm contents
import glob
precomp_files = sorted(glob.glob(os.path.join(PRECOMPUTED_DIR, "noised_r*.npz")))
print(f"\nPrecomputed repeats: {len(precomp_files)}")
if precomp_files:
    s = np.load(precomp_files[0])
    print(f"  x_t:    {s['x_t'].shape}  ({s['x_t'].dtype})")
    print(f"  s_true: {s['s_true'].shape}")
    print(f"  t:      {s['t'].shape},  range [{s['t'].min():.2f}, {s['t'].max():.2f}]")

## 2. Build manifold and SDE

In [ ]:
import jax
import jax.numpy as jnp

manifold = ShapeManifold(dim=d, numpoints=n, alpha=1.0, base=x_ref)
sde      = ManifoldVP(manifold)

# Sanity check: s_exp round-trip at t=0.5
x0_s = jnp.array(test_data[0:1, None])
rng  = jax.random.PRNGKey(0)
x_t_s, _, _ = sde.marginal_prob(x0_s, 0.5, rng)
dist     = float(manifold.s_distance(x0_s, x_t_s)[0, 0, 0])
expected = float(sde.alpha(0.5) * sde.sigma(0.5))
print(f"w^delta(x_t, x_0) = {dist:.4f}  (expected ~{expected:.4f})")
print("Manifold + SDE OK")

## 3. Benchmark Phase B on GPU

Phase A (geodesic noising) is skipped — data is precomputed. This benchmarks the JIT-compiled
gradient step alone and projects total training time.

In [ ]:
import time
import optax
from training.train_manifold import make_train_step
from training.score_loss import riemannian_dsm_loss_from_noised

BATCH_SIZE  = 64
LR          = 3e-4
HIDDEN_DIMS = (256, 256, 256, 256)

model     = TangentScoreModel(hidden_dims=HIDDEN_DIMS)
optimizer = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(LR))
train_step = make_train_step(model, manifold, sde, optimizer)

rng = jax.random.PRNGKey(42)
rng, init_key = jax.random.split(rng)
params    = model.init(init_key, jnp.zeros((1, n*d)), jnp.zeros((1, 1)))
ema_p     = params
opt_state = optimizer.init(params)

# Load one precomputed batch for benchmarking
s = np.load(precomp_files[0])
x_t_b   = jnp.array(s["x_t"][:BATCH_SIZE])
s_true_b = jnp.array(s["s_true"][:BATCH_SIZE])
t_b      = jnp.array(s["t"][:BATCH_SIZE])

print("Warming up JIT...")
params, ema_p, opt_state, loss = train_step(params, ema_p, opt_state, x_t_b, s_true_b, t_b)
jax.block_until_ready(loss)
print(f"  warm-up loss: {float(loss):.4f}")

times = []
for _ in range(20):
    t0 = time.time()
    params, ema_p, opt_state, loss = train_step(params, ema_p, opt_state, x_t_b, s_true_b, t_b)
    jax.block_until_ready(loss)
    times.append(time.time() - t0)

mean_ms = np.mean(times) * 1000
N_train = s["x_t"].shape[0]
n_steps_per_epoch = N_train // BATCH_SIZE
secs_per_epoch = np.mean(times) * n_steps_per_epoch
print(f"\nPhase B (grad step, B={BATCH_SIZE}): {mean_ms:.2f} ms/step")
print(f"Steps/epoch: {n_steps_per_epoch}  →  {secs_per_epoch:.1f}s/epoch")
print(f"Est. 3000 epochs: {3000*secs_per_epoch/60:.1f} min")

## 4. Train

In [23]:
import json
from training.train_manifold import train_from_precomputed

os.makedirs(CKPT_DIR, exist_ok=True)

N_EPOCHS    = 3000
BATCH_SIZE  = 64
LR          = 3e-4
HIDDEN_DIMS = (256, 256, 256, 256)
LOG_EVERY   = 100

model = TangentScoreModel(hidden_dims=HIDDEN_DIMS)

print(f"Training (precomputed): {N_EPOCHS} epochs, B={BATCH_SIZE}, lr={LR}")
print(f"Checkpoints → {CKPT_DIR}\n")

state, history = train_from_precomputed(
    model=model,
    manifold=manifold,
    sde=sde,
    precomputed_dir=PRECOMPUTED_DIR,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    grad_clip=1.0,
    ema_decay=0.995,
    likelihood_weighting=True,
    seed=42,
    log_every=LOG_EVERY,
    ckpt_dir=CKPT_DIR,
)

with open(os.path.join(CKPT_DIR, "loss_history.json"), "w") as f:
    json.dump(history, f)

first_loss = history[0][1]
last_loss  = history[-1][1]
print(f"\nInitial: {first_loss:.4f}  →  Final: {last_loss:.4f}  ({last_loss/first_loss*100:.1f}% of initial)")

Training (precomputed): 3000 epochs, B=64, lr=0.0003
Checkpoints → /content/runs/bba_phase36



NameError: name 'PRECOMPUTED_DIR' is not defined

## 5. Loss curve

In [ ]:
import matplotlib.pyplot as plt

epochs = [ep   for ep, _    in history]
losses = [loss for _,  loss in history]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, losses, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("DSM Loss")
ax.set_title(f"BBA Riemannian DSM — TangentScoreModel{HIDDEN_DIMS}, n=28 Cα")
ax.axhline(last_loss, color="r", linestyle="--", alpha=0.5,
           label=f"Final {last_loss:.2f} ({last_loss/first_loss*100:.0f}% of initial)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "loss_curve.png"), dpi=150)
plt.show()

## 6. Score sanity check

Cosine similarity between predicted and true score on held-out test frames.  
Expect > 0.5 after convergence.

In [ ]:
params   = state["ema_params"]
score_fn = lambda x_flat, t_col: model.apply(params, x_flat, t_col)

B_test = 64
rng    = jax.random.PRNGKey(99)
rng, t_key, n_key = jax.random.split(rng, 3)

x0_test = jnp.array(test_data[:B_test])
t_test  = jax.random.uniform(t_key, (B_test,), minval=0.2, maxval=0.8)
x_t, s_true = prepare_batch(manifold, sde, x0_test, t_test, n_key)

s_pred = score_fn(x_t.reshape(B_test, n*d), t_test.reshape(B_test, 1)).reshape(B_test, n, d)

a = s_pred.reshape(B_test, -1)
b = s_true.reshape(B_test, -1)
cos_sims = (a * b).sum(-1) / (jnp.linalg.norm(a, axis=-1) * jnp.linalg.norm(b, axis=-1) + 1e-8)

print(f"Score cosine similarity (t ∈ [0.2, 0.8]):")
print(f"  mean = {float(cos_sims.mean()):.4f}")
print(f"  std  = {float(cos_sims.std()):.4f}")
print(f"  min  = {float(cos_sims.min()):.4f}")

## 7. Save checkpoint to Google Drive

Colab VMs are ephemeral — save to Drive before the session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_OUT = "/content/drive/MyDrive/bba_phase36"  # <-- edit if needed
os.makedirs(DRIVE_OUT, exist_ok=True)

import shutil
for fname in ["ckpt_final.npz", "loss_history.json", "loss_curve.png"]:
    src = os.path.join(CKPT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_OUT, fname))
        print(f"Saved {fname}")